# MongoDB Airbnb to HDFS Parquet

Reads MongoDB Atlas `sample_airbnb.listingsAndReviews`, writes an ingest-date-partitioned raw HDFS Parquet copy, writes a flattened Hive-ready clean dataset, and builds a market-level analytics summary.

## 1. Load Runtime Configuration

This cell reads the environment variables passed by Docker Compose. It validates the MongoDB URI, chooses the raw ingest date, builds the raw partition path, and prints the HDFS output locations that the job will write.

In [ ]:
import os
from datetime import date

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F, types as T

mongo_uri = os.getenv("MONGODB_URI", "").strip()
mongo_database = os.getenv("MONGODB_DATABASE", "sample_airbnb")
mongo_collection = os.getenv("MONGODB_COLLECTION", "listingsAndReviews")
raw_path = os.getenv("AIRBNB_RAW_PATH", "hdfs://namenode:9000/data/airbnb/raw/listingsAndReviews")
configured_ingest_date = os.getenv("AIRBNB_INGEST_DATE", "").strip()
ingest_date = configured_ingest_date or date.today().isoformat()
date.fromisoformat(ingest_date)
raw_full_refresh = os.getenv("AIRBNB_RAW_FULL_REFRESH", "false").strip().lower() in {
    "1",
    "true",
    "yes",
    "y",
}
raw_partition_path = f"{raw_path.rstrip('/')}/ingest_date={ingest_date}"
clean_path = os.getenv("AIRBNB_CLEAN_PATH", "hdfs://namenode:9000/data/airbnb/clean/listings")
analytics_path = os.getenv(
    "AIRBNB_ANALYTICS_PATH",
    "hdfs://namenode:9000/data/airbnb/analytics/market_room_type_summary",
)
spark_master = os.getenv("SPARK_MASTER_URL", os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
connector_package = os.getenv(
    "MONGO_SPARK_CONNECTOR_PACKAGE",
    "org.mongodb.spark:mongo-spark-connector_2.12:10.2.2",
)
output_partitions = int(os.getenv("SPARK_OUTPUT_PARTITIONS", "4"))
shuffle_partitions = os.getenv("SPARK_SQL_SHUFFLE_PARTITIONS", "8")

if not mongo_uri:
    raise ValueError("MONGODB_URI is required. Copy env.example to .env and set the Atlas URI.")

print(f"Input collection: {mongo_database}.{mongo_collection}")
print(f"Raw output: {raw_path}")
print(f"Raw ingest partition: ingest_date={ingest_date}")
print(f"Clean output: {clean_path}")
print(f"Analytics output: {analytics_path}")

## 2. Create the Spark Session

This cell starts the Spark application against the Docker Spark master and loads the MongoDB Spark Connector. It also enables the local optimization settings used by the job: Adaptive Query Execution, Kryo serialization, compressed shuffle/spill, LZ4 internal compression, and Snappy Parquet output.

In [ ]:
spark = (
    SparkSession.builder.appName("mongo-airbnb-to-hdfs-parquet")
    .master(spark_master)
    .config("spark.jars.packages", connector_package)
    .config("spark.mongodb.read.connection.uri", mongo_uri)
    .config("spark.mongodb.read.database", mongo_database)
    .config("spark.mongodb.read.collection", mongo_collection)
    .config("spark.sql.shuffle.partitions", shuffle_partitions)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.shuffle.compress", "true")
    .config("spark.shuffle.spill.compress", "true")
    .config("spark.io.compression.codec", "lz4")
    .config("spark.sql.files.maxPartitionBytes", "128MB")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

## 3. Read MongoDB into the Raw DataFrame

This cell reads the source collection from MongoDB Atlas. The DataFrame is persisted with `MEMORY_AND_DISK` because the pipeline reuses it for raw writes, clean transformations, and row-count validation.

In [ ]:
raw_df = (
    spark.read.format("mongodb")
    .option("connection.uri", mongo_uri)
    .option("database", mongo_database)
    .option("collection", mongo_collection)
    .load()
)

raw_df = raw_df.persist(StorageLevel.MEMORY_AND_DISK)
raw_count = raw_df.count()
print(f"Read {raw_count:,} MongoDB documents.")
raw_df.printSchema()

## 4. Define Schema-Safe Projection Helpers

MongoDB documents can have nested and occasionally missing fields. These helper functions let the clean projection safely read nested fields, cast types, normalize booleans, flatten arrays, and extract coordinates without failing when an optional field is absent.

In [ ]:
def field_type(schema, path):
    current = schema
    for part in path.split("."):
        if not isinstance(current, T.StructType):
            return None
        try:
            field = current[part]
        except KeyError:
            return None
        current = field.dataType
    return current


def has_field(path):
    return field_type(raw_df.schema, path) is not None


def col_or_null(path, data_type="string"):
    if has_field(path):
        return F.col(path)
    return F.lit(None).cast(data_type)


def as_string(path):
    return col_or_null(path).cast("string")


def as_int(path):
    return col_or_null(path).cast("int")


def as_double(path):
    return col_or_null(path).cast("double")


def as_date(path):
    return F.to_date(col_or_null(path))


def as_bool(path):
    value = col_or_null(path).cast("string")
    lowered = F.lower(value)
    return (
        F.when(lowered.isin("true", "t", "yes", "y", "1"), F.lit(True))
        .when(lowered.isin("false", "f", "no", "n", "0"), F.lit(False))
        .otherwise(col_or_null(path).cast("boolean"))
    )


def array_csv(path):
    data_type = field_type(raw_df.schema, path)
    if isinstance(data_type, T.ArrayType):
        return F.concat_ws(",", F.col(path))
    return col_or_null(path).cast("string")


def array_size(path):
    data_type = field_type(raw_df.schema, path)
    if isinstance(data_type, T.ArrayType):
        value = F.col(path)
        return F.when(value.isNotNull(), F.size(value)).otherwise(F.lit(None).cast("int"))
    return F.lit(None).cast("int")


def coordinate(index):
    data_type = field_type(raw_df.schema, "address.location.coordinates")
    if isinstance(data_type, T.ArrayType):
        return F.col("address.location.coordinates").getItem(index).cast("double")
    return F.lit(None).cast("double")


listing_id = F.coalesce(as_string("_id.oid"), as_string("_id"))
country_code = F.upper(F.coalesce(as_string("address.country_code"), F.lit("unknown")))

## 5. Build the Hive-Ready Clean Listings Table

This cell flattens the nested Airbnb document into scalar columns that Hive can query easily. It extracts listing, host, address, review, availability, price, and location fields, then adds `country_code` as the clean table partition column.

In [ ]:
clean_df = raw_df.select(
    listing_id.alias("listing_id"),
    as_string("listing_url").alias("listing_url"),
    as_string("name").alias("name"),
    as_string("summary").alias("summary"),
    as_string("description").alias("description"),
    as_string("property_type").alias("property_type"),
    as_string("room_type").alias("room_type"),
    as_string("bed_type").alias("bed_type"),
    as_int("minimum_nights").alias("minimum_nights"),
    as_int("maximum_nights").alias("maximum_nights"),
    as_int("accommodates").alias("accommodates"),
    as_double("bedrooms").alias("bedrooms"),
    as_double("bathrooms").alias("bathrooms"),
    as_double("beds").alias("beds"),
    as_double("price").alias("price"),
    as_double("weekly_price").alias("weekly_price"),
    as_double("monthly_price").alias("monthly_price"),
    as_double("security_deposit").alias("security_deposit"),
    as_double("cleaning_fee").alias("cleaning_fee"),
    as_double("extra_people").alias("extra_people"),
    as_int("guests_included").alias("guests_included"),
    as_int("number_of_reviews").alias("number_of_reviews"),
    as_date("first_review").alias("first_review"),
    as_date("last_review").alias("last_review"),
    as_date("last_scraped").alias("last_scraped"),
    as_date("calendar_last_scraped").alias("calendar_last_scraped"),
    as_string("cancellation_policy").alias("cancellation_policy"),
    array_csv("amenities").alias("amenities_csv"),
    as_string("host.host_id").alias("host_id"),
    as_string("host.host_name").alias("host_name"),
    as_string("host.host_location").alias("host_location"),
    as_string("host.host_response_time").alias("host_response_time"),
    as_double("host.host_response_rate").alias("host_response_rate"),
    as_bool("host.host_is_superhost").alias("host_is_superhost"),
    as_int("host.host_listings_count").alias("host_listings_count"),
    as_int("host.host_total_listings_count").alias("host_total_listings_count"),
    as_string("host.host_neighbourhood").alias("host_neighbourhood"),
    as_bool("host.host_has_profile_pic").alias("host_has_profile_pic"),
    as_bool("host.host_identity_verified").alias("host_identity_verified"),
    as_string("address.street").alias("street"),
    as_string("address.suburb").alias("suburb"),
    as_string("address.government_area").alias("government_area"),
    as_string("address.market").alias("market"),
    as_string("address.country").alias("country"),
    coordinate(1).alias("latitude"),
    coordinate(0).alias("longitude"),
    as_bool("address.location.is_location_exact").alias("is_location_exact"),
    as_int("review_scores.review_scores_accuracy").alias("review_scores_accuracy"),
    as_int("review_scores.review_scores_cleanliness").alias("review_scores_cleanliness"),
    as_int("review_scores.review_scores_checkin").alias("review_scores_checkin"),
    as_int("review_scores.review_scores_communication").alias("review_scores_communication"),
    as_int("review_scores.review_scores_location").alias("review_scores_location"),
    as_int("review_scores.review_scores_value").alias("review_scores_value"),
    as_int("review_scores.review_scores_rating").alias("review_scores_rating"),
    as_int("availability.availability_30").alias("availability_30"),
    as_int("availability.availability_60").alias("availability_60"),
    as_int("availability.availability_90").alias("availability_90"),
    as_int("availability.availability_365").alias("availability_365"),
    F.coalesce(array_size("reviews"), as_int("number_of_reviews")).alias("reviews_count"),
    F.current_timestamp().alias("ingested_at"),
    country_code.alias("country_code"),
)

clean_df = clean_df.persist(StorageLevel.MEMORY_AND_DISK)
clean_df.printSchema()

## 6. Write Raw and Clean Parquet Outputs

This cell writes the raw layer under `ingest_date=YYYY-MM-DD` and writes the clean listings table partitioned by `country_code`. Normal raw writes replace only the selected ingest-date partition; `AIRBNB_RAW_FULL_REFRESH=true` rewrites the full raw table into the partitioned layout.

In [ ]:
if raw_full_refresh:
    raw_write_path = raw_path
    (
        raw_df.withColumn("ingest_date", F.lit(ingest_date))
        .coalesce(output_partitions)
        .write.mode("overwrite")
        .option("compression", "snappy")
        .partitionBy("ingest_date")
        .parquet(raw_path)
    )
else:
    raw_write_path = raw_partition_path
    (
        raw_df.coalesce(output_partitions)
        .write.mode("overwrite")
        .option("compression", "snappy")
        .parquet(raw_partition_path)
    )

(
    clean_df.repartition(output_partitions, "country_code")
    .sortWithinPartitions("country_code", "market", "listing_id")
    .write.mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("country_code")
    .parquet(clean_path)
)

clean_count = clean_df.count()
print(f"Wrote {raw_count:,} raw rows to {raw_write_path}")
print(f"Wrote {clean_count:,} clean rows to {clean_path}")

## 7. Build the Market Analytics Summary

This cell creates a business-ready aggregate by country, market, room type, and property type. It calculates pricing, demand, review-quality, superhost, and host-verification metrics, then writes the result as a Hive-ready Parquet table partitioned by `country_code`.

In [ ]:
price_per_guest = F.when(F.col("accommodates") > 0, F.col("price") / F.col("accommodates"))
availability_rate_365 = F.when(
    F.col("availability_365").isNotNull(), F.col("availability_365") / F.lit(365.0)
)
review_score_cols = [
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
]
review_score_sum = sum(F.coalesce(F.col(col_name).cast("double"), F.lit(0.0)) for col_name in review_score_cols)
review_score_count = sum(
    F.when(F.col(col_name).isNotNull(), F.lit(1.0)).otherwise(F.lit(0.0))
    for col_name in review_score_cols
)
review_score_avg = F.when(review_score_count > 0, review_score_sum / review_score_count)

listing_features_df = clean_df.select(
    "country_code",
    F.coalesce(F.col("country"), F.lit("unknown")).alias("country"),
    F.coalesce(F.col("market"), F.lit("unknown")).alias("market"),
    F.coalesce(F.col("room_type"), F.lit("unknown")).alias("room_type"),
    F.coalesce(F.col("property_type"), F.lit("unknown")).alias("property_type"),
    "listing_id",
    "host_id",
    "price",
    price_per_guest.alias("price_per_guest"),
    F.when(F.col("reviews_count") > 0, F.col("review_scores_rating") / F.lit(20.0)).alias(
        "rating_stars"
    ),
    review_score_avg.alias("review_score_avg"),
    availability_rate_365.alias("availability_rate_365"),
    (F.lit(1.0) - availability_rate_365).alias("booking_pressure_score"),
    F.when(F.col("host_is_superhost") == True, F.lit(1.0)).otherwise(F.lit(0.0)).alias(
        "superhost_flag"
    ),
    F.when(F.col("host_identity_verified") == True, F.lit(1.0)).otherwise(F.lit(0.0)).alias(
        "host_verified_flag"
    ),
    "reviews_count",
)

analytics_df = listing_features_df.groupBy(
    "country_code", "country", "market", "room_type", "property_type"
).agg(
    F.count("listing_id").alias("listing_count"),
    F.countDistinct("host_id").alias("host_count"),
    F.round(F.avg("price"), 2).alias("avg_price"),
    F.expr("percentile_approx(price, 0.5, 100)").alias("median_price"),
    F.round(F.avg("price_per_guest"), 2).alias("avg_price_per_guest"),
    F.round(F.avg("rating_stars"), 2).alias("avg_rating_stars"),
    F.round(F.avg("review_score_avg"), 2).alias("avg_review_score"),
    F.round(F.avg("availability_rate_365"), 4).alias("availability_rate_365"),
    F.round(F.avg("booking_pressure_score"), 4).alias("booking_pressure_score"),
    F.round(F.avg("superhost_flag"), 4).alias("superhost_rate"),
    F.round(F.avg("host_verified_flag"), 4).alias("host_verified_rate"),
    F.sum("reviews_count").cast("int").alias("total_reviews"),
).filter(F.col("listing_count") >= 1).persist(StorageLevel.MEMORY_AND_DISK)

(
    analytics_df.repartition(output_partitions, "country_code")
    .sortWithinPartitions("country_code", "market", "room_type", "property_type")
    .write.mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("country_code")
    .parquet(analytics_path)
)

analytics_count = analytics_df.count()
print(f"Wrote {analytics_count:,} market summary rows to {analytics_path}")

## 8. Release Spark Resources

This cell unpersists cached DataFrames and stops the Spark session so executors and driver resources are released cleanly.

In [ ]:
analytics_df.unpersist()
clean_df.unpersist()
raw_df.unpersist()
spark.stop()